In [16]:
import os
import glob

import pandas as pd

In [17]:
mapping = pd.read_csv(
    "../datasets/reference/crop_name_mapping.csv"
)

mapping.head()

,Price_Commodity,Standard_Crop
0,Apple,Apple
1,Arecanut (Betelnut/Supari),Arecanut
2,Arhar (Tur/Red Gram)(Whole),Arhar/Tur
3,Pegeon Pea (Arhar Fali),Arhar/Tur
4,Red Gram,Arhar/Tur


In [18]:
print(mapping.shape)
mapping.info()

(89, 2)
<class 'pandas.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Price_Commodity  89 non-null     str  
 1   Standard_Crop    89 non-null     str  
dtypes: str(2)
memory usage: 3.3 KB


In [19]:
files = sorted(
    glob.glob("../datasets/cleaned/crop_price/*.parquet")
)

len(files)

26

In [20]:
def process_price_file(file_path):

    df = pd.read_parquet(file_path)

    # Merge commodity names
    df = df.merge(
        mapping,
        left_on="Commodity",
        right_on="Price_Commodity",
        how="inner"
    )

    # Keep only required columns
    df = df[
        [
            "State",
            "District",
            "Market",
            "Arrival_Date",
            "Year",
            "Month",
            "Standard_Crop",
            "Modal_Price"
        ]
    ]

    return df

In [21]:
frames = []

for file in files:

    print(f"Processing {os.path.basename(file)}")

    yearly = process_price_file(file)

    frames.append(yearly)

Processing 2001.parquet
Processing 2002.parquet
Processing 2003.parquet
Processing 2004.parquet
Processing 2005.parquet
Processing 2006.parquet
Processing 2007.parquet
Processing 2008.parquet
Processing 2009.parquet
Processing 2010.parquet
Processing 2011.parquet
Processing 2012.parquet
Processing 2013.parquet
Processing 2014.parquet
Processing 2015.parquet
Processing 2016.parquet
Processing 2017.parquet
Processing 2018.parquet
Processing 2019.parquet
Processing 2020.parquet
Processing 2021.parquet
Processing 2022.parquet
Processing 2023.parquet
Processing 2024.parquet
Processing 2025.parquet
Processing 2026.parquet


In [22]:
price_df = pd.concat(
    frames,
    ignore_index=True
)

In [23]:
price_df.shape

(52760648, 8)

In [24]:
price_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52760648 entries, 0 to 52760647
Data columns (total 8 columns):
 #   Column         Dtype         
---  ------         -----         
 0   State          str           
 1   District       str           
 2   Market         str           
 3   Arrival_Date   datetime64[us]
 4   Year           int32         
 5   Month          int32         
 6   Standard_Crop  str           
 7   Modal_Price    float64       
dtypes: datetime64[us](1), float64(1), int32(2), str(4)
memory usage: 4.5 GB


In [25]:
price_df.head()

,State,District,Market,Arrival_Date,Year,Month,Standard_Crop,Modal_Price
0,Karnataka,Dharwad,Hubli (Amaragol),2001-01-10,2001,1,Gram,1400.0
1,Karnataka,Dharwad,Hubli (Amaragol),2001-01-10,2001,1,Blackgram,1380.0
2,Karnataka,Dharwad,Hubli (Amaragol),2001-01-10,2001,1,Moong(Green Gram),1800.0
3,Karnataka,Dharwad,Hubli (Amaragol),2001-01-10,2001,1,Groundnut,1105.0
4,Karnataka,Dharwad,Hubli (Amaragol),2001-01-10,2001,1,Maize,440.0


In [26]:
price_df.isnull().sum()

State            0
District         0
Market           0
Arrival_Date     0
Year             0
Month            0
Standard_Crop    0
Modal_Price      0
dtype: int64

In [27]:
price_df.to_parquet(
    "../datasets/processed/mapped_crop_prices.parquet",
    index=False
)

price_df.to_csv(
    "../datasets/processed/mapped_crop_prices.csv",
    index=False
)